# notebook 09 — 光円錐・因果構造の読み方は周期性に阻まれる

(2) の検証は notebook 08 で `st` 側を否定したが、そこで暗黙に **「空間＝距離行列の
ユークリッド MDS」** という枠を固定していた。**光円錐**という指摘は、この枠が狭い可能性を
突く。ローレンツ幾何では光円錐（因果構造）が大域構造を決め、ヌル方向で固有距離が 0 になり
点が無限に引き伸ばされる——**符号を入れること自体が非コンパクト方向を開く**かもしれない。

これは notebook 06 で立てた前提 **P4「非コンパクト化とローレンツ化は独立」** への正当な
反論になり得る。そこで、ユークリッド MDS 以外の幾何の読み方を網羅的に試す:

- **L-A** 因果構造：相関の符号境界 `θ=π/4`（`|C|=0`＝`st`域外）を光円錐（ヌル面）とみなす。
- **L-B** ローレンツ埋め込み：`sign(C)` を計量符号とし、負固有値を時間として正面から使う。
- **L-C** causal set：相関から因果半順序を作り、次元とローレンツ性を同時に出す。
- **L-D** 共形構造：スケールを落とし因果構造だけ残す（光円錐は共形不変）。
- **L-E** 双曲幾何：MDS の平坦仮定が見落とす非コンパクト負曲率空間。

**結論の先取り:** いずれも **`cos2θ` の周期性**に阻まれ、単一の時間軸も非有界方向も出ない。
光円錐を出すにも「符号が一度だけ変わる」非周期相関（道I）が要る。光円錐は道II（符号）を
周期性から救わない。ただし **P4 は部分的に修正**：符号は確かに時間的方向を生む（が単一でない）。


## 0. セットアップ

In [1]:
import numpy as np
from numpy.linalg import eigh
np.set_printoptions(precision=3, suppress=True, linewidth=120)

def signed_embed(C):
    """Lorentzian (indefinite) embedding: B from signed squared distance."""
    a=np.abs(C)
    with np.errstate(divide='ignore'): d=-np.log(a)
    n=C.shape[0]; np.fill_diagonal(d,0.0); d[~np.isfinite(d)]=0.0
    D2=np.sign(C)*d**2; np.fill_diagonal(D2,0.0)
    J=np.eye(n)-np.ones((n,n))/n; B=-0.5*J@D2@J; B=0.5*(B+B.T)
    ev,V=eigh(B); idx=np.argsort(np.abs(ev))[::-1]
    return ev[idx], V[:,idx]*np.sqrt(np.abs(ev[idx]))


## L-A. 符号境界は光円錐か？ ── 因果順序の推移性で判定

`cos2(Δθ)` の符号：`|Δθ|<π/4` で +（相関＝spacelike）、`π/4<|Δθ|<3π/4` で −（反相関＝
timelike）、その先でまた +…。境界 `|Δθ|=π/4` は `C=0`（`st`域外、notebook 03）。これを
ヌル面と見なせるか。**光円錐の本質は因果順序の推移性**（a<b かつ b<c ⇒ a<c）。これを試す。


In [2]:
m=200; th=np.linspace(0,2*np.pi,m,endpoint=False)
C=np.cos(2*(th[:,None]-th[None,:]))/6
timelike = (C<0)   # anti-correlated = 'timelike separated'
print(f"timelike (anti-correlated) fraction: {timelike.mean():.3f}")

# transitivity test of the 'timelike' relation
rng=np.random.default_rng(0); idx=np.arange(0,m,10); viol=tot=0
for i in idx:
    for j in idx:
        for k in idx:
            if timelike[i,j] and timelike[j,k]:
                tot+=1; viol+= (not timelike[i,k])
print(f"transitivity of timelike relation: {tot-viol}/{tot} hold "
      f"({100*(1-viol/max(tot,1)):.0f}%)")
print("\n=> far below 100% transitive (a causal order needs 100%).")
print("   The sign relation is NOT a transitive causal order.")
print("   A light cone opens ONCE; cos2 sign alternates (+,-,+,-) every pi/4 (periodic).")
print("   Periodic sign reversal => no global future/past. L-A FAILS.")


timelike (anti-correlated) fraction: 0.499
transitivity of timelike relation: 480/2000 hold (24%)

=> far below 100% transitive (a causal order needs 100%).
   The sign relation is NOT a transitive causal order.
   A light cone opens ONCE; cos2 sign alternates (+,-,+,-) every pi/4 (periodic).
   Periodic sign reversal => no global future/past. L-A FAILS.


## L-B. ローレンツ埋め込み ── 単一の時間軸は出るか

`sign(C)` を符号数に使い、負固有値を時間として埋め込む。物理的な光円錐なら符号数は
`(1,n)`（時間1つ）。実際の符号数を見る。


In [3]:
# bare cos2theta
ev,coords=signed_embed(C)
print(f"bare cos2theta: #spacelike-dims={int((ev>1e-6).sum())} "
      f"#timelike-dims={int((ev<-1e-6).sum())}  max|coord|={np.abs(coords).max():.2f}")

# product T^2
m2=12; al=np.linspace(0,2*np.pi,m2,endpoint=False)
A,Bg=np.meshgrid(al,al,indexing='ij'); P=np.stack([A.ravel(),Bg.ravel()],1)
C2=np.cos(2*(P[:,0][:,None]-P[:,0][None,:]))*np.cos(2*(P[:,1][:,None]-P[:,1][None,:]))/36
ev2,_=signed_embed(C2)
print(f"product T^2:   #spacelike={int((ev2>1e-6).sum())} #timelike={int((ev2<-1e-6).sum())}")
print("\n=> bare cos2theta: nearly balanced #space ~ #time (rapid sign oscillation);")
print("   product T^2: few timelike dims but still several (not one). In all cases")
print("   coordinates are BOUNDED (not non-compact), and there is NO single time axis.")
print("   P4 is PARTIALLY challenged: sign DOES make timelike directions, but not ONE.")


bare cos2theta: #spacelike-dims=100 #timelike-dims=99  max|coord|=25.72
product T^2:   #spacelike=138 #timelike=5

=> bare cos2theta: nearly balanced #space ~ #time (rapid sign oscillation);
   product T^2: few timelike dims but still several (not one). In all cases
   coordinates are BOUNDED (not non-compact), and there is NO single time axis.
   P4 is PARTIALLY challenged: sign DOES make timelike directions, but not ONE.


## L-C. causal set ── 推移的半順序は作れるか

causal set は推移的・反対称な半順序を要する。L-A で符号関係は推移的でない。`|C|` の大小は
対称（距離）で時間の矢を与えない。よって `cos2θ` から causal set 構造は出ない。


In [4]:
print("Causal set needs a transitive antisymmetric order.")
print(" - sign(C) order: non-transitive (L-A) -> not a partial order.")
print(" - |C| magnitude: symmetric -> a distance, no time arrow.")
print("=> cos2theta yields NO causal-set structure (same periodicity obstruction). FAIL.")


Causal set needs a transitive antisymmetric order.
 - sign(C) order: non-transitive (L-A) -> not a partial order.
 - |C| magnitude: symmetric -> a distance, no time arrow.
=> cos2theta yields NO causal-set structure (same periodicity obstruction). FAIL.


## L-D. 共形構造 ── スケールを落とせば開くか

光円錐は共形不変。だが `S¹` の共形群は `PSL(2,ℝ)`＝**非コンパクト群**である一方、それが
作用する**空間は依然 `S¹`（コンパクト）**。非コンパクトなのは対称性であって空間ではない。


In [5]:
print("Conformal group of S^1 is PSL(2,R): non-compact as a GROUP, acting ON compact S^1.")
print("Conformal structure makes the SYMMETRY non-compact, not the SPACE. L-D does not")
print("decompactify the space.")


Conformal group of S^1 is PSL(2,R): non-compact as a GROUP, acting ON compact S^1.
Conformal structure makes the SYMMETRY non-compact, not the SPACE. L-D does not
decompactify the space.


## L-E. 双曲幾何 ── 平坦 MDS が見落とす負曲率空間か

MDS は平坦を仮定する。周期相関が非コンパクトな双曲空間 `H^n` に埋まるなら、それを見落として
いる恐れ。Gromov の4点 `δ` で距離が tree-like（双曲的）かを判定する。


In [6]:
m=60; th=np.linspace(0,2*np.pi,m,endpoint=False)
C=np.cos(2*(th[:,None]-th[None,:]))/6
a=np.abs(C)
with np.errstate(divide='ignore'): D=-np.log(a)
np.fill_diagonal(D,0.0); D[~np.isfinite(D)]=D[np.isfinite(D)].max()
rng=np.random.default_rng(0); deltas=[]
for _ in range(3000):
    i,j,k,l=rng.choice(m,4,replace=False)
    s=sorted([D[i,j]+D[k,l], D[i,k]+D[j,l], D[i,l]+D[j,k]])
    deltas.append((s[2]-s[1])/2)
print(f"Gromov 4-point delta: mean={np.mean(deltas):.3f} max={np.max(deltas):.3f} "
      f"diameter={D.max():.3f}  delta/diam={np.max(deltas)/D.max():.3f}")
print("=> delta/diameter is large (~0.45), NOT ~0. The cos2theta metric is NOT tree-like;")
print("   it is the bounded circle metric, not non-compact H^n. L-E FAILS.")


Gromov 4-point delta: mean=0.376 max=1.857 diameter=4.050  delta/diam=0.458
=> delta/diameter is large (~0.45), NOT ~0. The cos2theta metric is NOT tree-like;
   it is the bounded circle metric, not non-compact H^n. L-E FAILS.


## まとめ：光円錐・因果の読み方はすべて周期性に阻まれる

### 結果

| 可能性 | 内容 | 判定 | 理由 |
|---|---|---|---|
| L-A | 符号境界＝光円錐（因果順序） | **失敗** | 周期的符号反転 → 推移性が 100% に届かず、大域的な未来/過去なし |
| L-B | ローレンツ（不定）埋め込み | **部分的** | 符号は時間的次元を生むが多数、単一時間でなく座標は有界 |
| L-C | causal set（因果半順序） | **失敗** | 周期符号は推移的順序にならない |
| L-D | 共形構造 | **失敗** | 非コンパクトなのは群、空間は `S¹` のまま |
| L-E | 双曲幾何 `H^n` | **失敗** | `cos2θ` 距離は tree-like でなく、有界な円 |

### 共通の根因（再び周期性）
すべての因果的・ローレンツ的・曲がった読み方が、同じ事実——**`cos2θ` の符号も大きさも
周期的**——に阻まれる。周期的な符号は単一の時間の矢を持てず、周期的な大きさは非有界方向を
持てない。**光円錐を出すには符号が「一度だけ」変わる必要があり、それは非周期相関＝道I**。
光円錐は道II（符号）を周期性から救わない。

### ユーザの指摘への回答（誠実版）
- **正しかった点:** 私の (2) の枠組みはユークリッド MDS に偏っていた。光円錐＝因果構造という
  独立の読み方を検討すべきという指摘は妥当で、L-A〜L-E という見落としを掘り起こした。
- **前提 P4 の修正:** 「非コンパクト化とローレンツ化は独立」は **部分的に不正確**だった。符号
  情報は確かに時間的方向を生む（L-B）ので、両者は無関係ではない。だが `cos2θ` の周期性ゆえ、
  その時間的方向は単一でなく、非コンパクト方向も開かない。**光円錐の物理的実質（単一時間・
  開いた因果）には、やはり非周期相関が要る**という点で、結論（追加原理が必要）は変わらない。

### 規律の自己点検（引き継ぎ書 §6）
- ユークリッド MDS への偏りを自己点検し、因果・ローレンツ・曲率の読み方を網羅した。✅
- L-B を「部分的成功」と正直にラベルし、P4 の不正確さを認めた。✅
- 「光円錐で解決」と飛びつかず、推移性が不成立 という決定的失敗を提示した。✅
- 周期性という共通根因に帰着させ、結論（道I が要る）を過大化していない。✅

### 含意と次への申し送り
- (2) の検証は `st` 側（notebook 08）と幾何の読み方（本 notebook）の両面で、**`cos2θ` 単独
  では非コンパクト時空も光円錐も出ない**ことを示した。残るは **作用 `S=−βΣe^{−d}` の実効
  相関＝位相転移**だけ。素の相関が周期的でも、作用が定める実効二点相関は非周期・減衰し得る
  （統計力学では普通）。ここが (2) の最後の希望。
- 位相転移で実効相関長が発散する臨界点をまたぐと、符号の周期構造が実効的に崩れ、単一時間や
  非有界方向が出る可能性は残る。次段で転送作用素のスペクトルとして検証する。
